**FACILITATOR COPY** — example guesses supplied to allow complete execution. Students should independently choose their own five guesses. Never distribute this copy as the student version.

# Data Science Detective — Can You Outsmart the Algorithm?

**40 minutes | A-Level | No previous Python required**  
**Mission:** Your university has limited capacity to offer extra study support. Can you identify students who might need help *before* their final assessment?

You will move from **human intuition → hand-written rules → a trained decision tree** and check how well each approach works. All records are fictional.

**How to use this notebook:** Run a code cell with **Shift + Enter**. Only edit code where you see ✏️ **YOUR TURN**. Read the output before continuing.

**Privacy and fairness:** Predictions are clues, not judgements of students. Never use a prediction to deny support.

## 0–5 minutes · Meet the data

Each row contains a fictional student's attendance (%), independent study hours per week, earlier **assignment mark** (%), and the result of a **later** assessment (`Passed`: 1 = pass, 0 = did not pass). This earlier mark does **not** reveal the later result.

We will keep **50 records out of training** for a fair test. We will look at five of these without showing their results. **Do not peek at the answers yet!**

In [ ]:
import csv, random, math
from pathlib import Path

# Keep the CSV in the same JupyterLite folder as this notebook.
with open('student_data.csv', newline='') as f:
    students = [{key: (value if key == 'StudentID' else int(value))
                 for key, value in record.items()}
                for record in csv.DictReader(f)]

rng = random.Random(42)
rng.shuffle(students)
train, test = students[:150], students[150:]
print('Students:', len(students), '| Training:', len(train), '| Test:', len(test))
print('Example row (training data):', train[0])
print('Training pass rate:', round(100*sum(s['Passed'] for s in train)/len(train)), '%')

## Task 1 · Trust your instinct (5–10 minutes)

Before using any algorithm, consider the **five students below**. Each needs either `1 = likely to pass` or `0 = might need support`. You may discuss your guesses with a partner.

✏️ **YOUR TURN:** Replace the five `None` values in the next cell with `0` or `1`. Do **not** use the `Passed` value — it is hidden.

In [ ]:
challenge = test[:5]
for student in challenge:
    print(student['StudentID'], '| Attendance:', student['Attendance'],
          '| Study hours:', student['StudyHours'],
          '| Earlier assignment:', student['AssignmentMark'])

# ✏️ YOUR TURN: replace each None with 0 or 1, then run the cell again.
my_guesses = [1, 1, 0, 0, 1]
assert len(my_guesses) == 5 and all(x in (0, 1) for x in my_guesses),     'Enter five guesses using only 0 and 1.'
print('Your predictions are saved — keep going!')

## 10–18 minutes · A rule-based classifier

A **classifier** predicts a category. This first classifier uses conditions that **you choose**. It does not learn from the data. A score of `0` means the rule suggests the student might need support; `1` means likely to pass.

✏️ **Task 2:** Run the next cell. Then try changing `ATTENDANCE_LIMIT` from 70 to 80. How many test predictions change?

In [ ]:
# ✏️ YOUR TURN: change a number and rerun the cell.
ATTENDANCE_LIMIT = 70
MARK_LIMIT = 60

def rule_predict(student):
    if (student['Attendance'] >= ATTENDANCE_LIMIT and
        student['AssignmentMark'] >= MARK_LIMIT):
        return 1
    return 0

rule_predictions = [rule_predict(s) for s in test]
print('Rule predicts passes:', sum(rule_predictions), 'out of', len(test))
print('First five rule predictions:', rule_predictions[:5])

## 18–28 minutes · Let the computer learn a decision tree

A **decision tree** asks a sequence of questions, such as *Is the earlier assignment mark above a certain value?* Unlike the rule-based classifier, it selects questions and thresholds from **training data**.

To save time, the tree-building code is already written. You do **not** need to understand every line. Running it is enough. It uses the **Gini impurity** to choose splits and works in browser-based Python without installing `scikit-learn`.

In [ ]:
FEATURES = ['Attendance', 'StudyHours', 'AssignmentMark']

def gini(rows):
    p = sum(r['Passed'] for r in rows) / len(rows)
    return 2 * p * (1 - p)

def build_tree(rows, depth=0, max_depth=3, min_leaf=8):
    majority = int(sum(r['Passed'] for r in rows) >= len(rows)/2)
    node = {'prediction': majority, 'count': len(rows)}
    if depth >= max_depth or gini(rows) == 0:
        return node
    best = None
    for feature in FEATURES:
        values = sorted(set(r[feature] for r in rows))
        for a, b in zip(values, values[1:]):
            threshold = (a + b) / 2
            left = [r for r in rows if r[feature] <= threshold]
            right = [r for r in rows if r[feature] > threshold]
            if min(len(left), len(right)) < min_leaf:
                continue
            impurity = (len(left)*gini(left) + len(right)*gini(right))/len(rows)
            if best is None or impurity < best[0]:
                best = (impurity, feature, threshold, left, right)
    if best is None or best[0] >= gini(rows):
        return node
    _, feature, threshold, left, right = best
    node.update(feature=feature, threshold=threshold,
                left=build_tree(left, depth+1, max_depth, min_leaf),
                right=build_tree(right, depth+1, max_depth, min_leaf))
    return node

def tree_predict(student, node):
    while 'feature' in node:
        node = node['left'] if student[node['feature']] <= node['threshold'] else node['right']
    return node['prediction']

def show_tree(node, depth=0):
    indent = '  ' * depth
    if 'feature' not in node:
        print(indent + f"Predict {node['prediction']} ({node['count']} training records)")
        return
    print(indent + f"If {node['feature']} <= {node['threshold']:.1f}:")
    show_tree(node['left'], depth+1)
    print(indent + 'Else:')
    show_tree(node['right'], depth+1)

# Fit on training data ONLY. Evaluation will use the untouched test data.
tree = build_tree(train, max_depth=3)
tree_predictions = [tree_predict(s, tree) for s in test]
print('Learned decision tree:')
show_tree(tree)
print('\nFirst five tree predictions:', tree_predictions[:5])

### Task 3 · Investigate the learned rules

Look at the printed tree. Which feature does it check **first**? Does it use attendance, study hours or assignment mark?

✏️ **YOUR TURN:** Change `max_depth=3` to `max_depth=1` in the preceding cell and rerun it. How does the tree change? Return to 3 before proceeding if you want to compare the original model.

## 28–36 minutes · Was the computer right?

We can now reveal the results for the five students and evaluate both algorithms on the **same 50 previously unseen records**.

A **confusion matrix** counts four possible outcomes:
- **True positive (TP):** predicted *pass*, actually passed
- **False positive (FP):** predicted *pass*, actually did not pass
- **False negative (FN):** predicted *did not pass*, actually passed
- **True negative (TN):** predicted *did not pass*, actually did not pass

For a support programme, a **false positive** could mean missing someone who needs help. Think carefully about the consequences.

In [ ]:
def evaluate(actual, predicted):
    tp = sum(a == 1 and p == 1 for a, p in zip(actual, predicted))
    fp = sum(a == 0 and p == 1 for a, p in zip(actual, predicted))
    fn = sum(a == 1 and p == 0 for a, p in zip(actual, predicted))
    tn = sum(a == 0 and p == 0 for a, p in zip(actual, predicted))
    accuracy = (tp + tn) / len(actual)
    return tp, fp, fn, tn, accuracy

def show_evaluation(label, actual, predicted):
    tp, fp, fn, tn, accuracy = evaluate(actual, predicted)
    print('\n' + label)
    print('                Predicted PASS   Predicted NOT PASS')
    print(f'Actual PASS          {tp:3d}                 {fn:3d}')
    print(f'Actual NOT PASS      {fp:3d}                 {tn:3d}')
    print(f'Accuracy: {accuracy:.0%} | FP: {fp} | FN: {fn}')

actual = [s['Passed'] for s in test]
show_evaluation('RULE-BASED CLASSIFIER (50 test records)', actual, rule_predictions)
show_evaluation('DECISION TREE (same 50 test records)', actual, tree_predictions)
print('\nTHE FIVE STUDENTS:')
for s, guess, rule, machine in zip(challenge, my_guesses, rule_predictions, tree_predictions):
    print(s['StudentID'], 'Your guess:', guess, '| Rule:', rule,
          '| Tree:', machine, '| Actual:', s['Passed'])
show_evaluation('YOUR INSTINCTS (five students ONLY)',
                [s['Passed'] for s in challenge], my_guesses)

### Task 4 · Make the decision (36–40 minutes)

**Discuss (no extra code required):**

1. On the 50-record test, which algorithm has better accuracy? Which makes fewer **false positives**?
2. Does higher accuracy always mean a better way to decide who should receive support?
3. Why is comparing your five instinctive guesses with the models' 50-record accuracy **not** a fair overall comparison?

**Remember:** One small test cannot establish a trustworthy real-world model. The records are simulated and performance will vary with new data. Student support should be based on conversation and context, not solely on an algorithm.

### Where could this lead?

You have used the same ideas studied in Computer Science, Data Science, Statistics and Artificial Intelligence degrees. Relevant careers include **data analyst, data scientist, machine learning engineer** and **education researcher**.

**Takeaway:** Good data science means asking sensible questions, creating models, checking mistakes and deciding how to use predictions responsibly.